# Advanced Imputation Techniques & Target-Guided Encoding
Global mean and median imputation often fail to preserve natural feature relationships when dealing with complex missing data patterns. 

This notebook explores advanced feature engineering strategies on the Titanic dataset—focusing on regex string extraction, conditional groupby median filling, multi-feature KNN imputation, and target-guided ordinal encoding.

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import KNNImputer

### Load Dataset

In [3]:
df = pd.read_csv('/Users/rkp/Downloads/titanic.csv')

In [4]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


### Extract Title from Name column using Regular Expressions

In [6]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.',expand=False)

In [7]:
df.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S,Mr


### Conditional Median Imputation for 'Age' based on 'Pclass' and 'Title'

In [10]:
df['Age'] = df.groupby(['Pclass','Title'])['Age'].transform(lambda x : x.fillna(x.median()))
df['Age'] = df['Age'].fillna(df['Age'].median()) # Fallback for rare unmapped titles

### KNN Imputation for Numerical Features

In [16]:
knn_imputer = KNNImputer(n_neighbors=5)
numerical_col = ['Age', 'Fare', 'SibSp', 'Parch']
df[numerical_col] = knn_imputer.fit_transform(df[numerical_col])

### Target-Guided Ordinal Encoding on High-Cardinality 'Embarked'

In [18]:
embarked_target_mean = df.groupby('Embarked')['Survived'].mean()
df['Embarked_Target_Encoded'] = df['Embarked'].map(embarked_target_mean)

In [19]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Embarked_Target_Encoded
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0.0,A/5 21171,7.2500,NaN,S,Mr,0.336957
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC 17599,71.2833,C85,C,Mrs,0.553571
